# Replay-Based Market Microstructure Analysis

This notebook uses `mm_replay` to reconstruct order book state and (optionally) interleaved trades from a recorded day.

Use it for feature research such as:
- order book imbalance
- spread / depth regime changes
- signed trade flow
- short-horizon return response to imbalance / flow


## Setup

Adjust `DAY_DIR` / `EXCHANGE` / `TOP_N` below. For Binance examples, `TOP_N=20` or `100` works well.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mm_replay.orderbook import ReplayConfig, replay_orderbook_day

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


In [ ]:
# Example day (edit these)
DAY_DIR = Path('data/binance/BTCUSDT/20260221')
EXCHANGE = 'binance'
TOP_N = 20          # use 100 if your recorder stores 100 levels
INCLUDE_TRADES = True
ON_ERROR = 'strict' # or 'best-effort'

assert DAY_DIR.exists(), f'Missing day dir: {DAY_DIR}'


## Replay Into Memory

This collects replay frames (`book`, `trade`, and possibly `discontinuity`) into a Python list for analysis.


In [ ]:
frames = []

def _emit(frame: dict):
    frames.append(frame)

stats = replay_orderbook_day(
    ReplayConfig(
        day_dir=DAY_DIR,
        exchange=EXCHANGE,
        top_n=TOP_N,
        on_error=ON_ERROR,
        speed=0,
        time_base='recv',
        include_trades=INCLUDE_TRADES,
        validate_only=False,
    ),
    emit=_emit,
)

stats.as_dict(), len(frames)


In [ ]:
# Split by frame type
book_frames = [f for f in frames if f.get('type') == 'book']
trade_frames = [f for f in frames if f.get('type') == 'trade']
disc_frames = [f for f in frames if f.get('type') == 'discontinuity']

len(book_frames), len(trade_frames), len(disc_frames)


## Build DataFrames


In [ ]:
books = pd.DataFrame(book_frames)
trades = pd.DataFrame(trade_frames) if trade_frames else pd.DataFrame()
discs = pd.DataFrame(disc_frames) if disc_frames else pd.DataFrame()

if not books.empty:
    books = books.sort_values('recv_seq').reset_index(drop=True)
    books['recv_ts'] = pd.to_datetime(books['recv_ms'], unit='ms', utc=True)
    books['event_ts'] = pd.to_datetime(books['event_time_ms'], unit='ms', utc=True, errors='coerce')

if not trades.empty:
    trades = trades.sort_values('recv_seq').reset_index(drop=True)
    trades['recv_ts'] = pd.to_datetime(trades['recv_ms'], unit='ms', utc=True)
    trades['trade_ts'] = pd.to_datetime(trades['trade_time_ms'], unit='ms', utc=True, errors='coerce')

books.head(2), trades.head(2) if not trades.empty else 'no trades'


## Feature Engineering (Books)

We derive spread, mid, and top-N imbalance from replayed book frames.


In [ ]:
def _sum_side_qty(levels, n):
    if not isinstance(levels, list):
        return np.nan
    total = 0.0
    for lvl in levels[:n]:
        if not lvl or len(lvl) < 2:
            continue
        total += float(lvl[1])
    return total

def _book_imbalance(row, n=5):
    b = _sum_side_qty(row.get('bids'), n)
    a = _sum_side_qty(row.get('asks'), n)
    den = b + a
    if pd.isna(den) or den == 0:
        return np.nan
    return (b - a) / den

if not books.empty:
    books['best_bid'] = pd.to_numeric(books['best_bid'], errors='coerce')
    books['best_ask'] = pd.to_numeric(books['best_ask'], errors='coerce')
    books['mid'] = (books['best_bid'] + books['best_ask']) / 2.0
    books['spread'] = books['best_ask'] - books['best_bid']
    books['imbalance_l1'] = books.apply(lambda r: _book_imbalance(r, n=1), axis=1)
    books['imbalance_l5'] = books.apply(lambda r: _book_imbalance(r, n=min(5, TOP_N)), axis=1)
    books['imbalance_l20'] = books.apply(lambda r: _book_imbalance(r, n=min(20, TOP_N)), axis=1)

books[['recv_seq', 'recv_ts', 'best_bid', 'best_ask', 'mid', 'spread', 'imbalance_l1', 'imbalance_l5', 'imbalance_l20']].head()


## Feature Engineering (Trades)

Signed trade flow uses replay-normalized `side` (`buy` => positive, `sell` => negative).


In [ ]:
if not trades.empty:
    trades['price'] = pd.to_numeric(trades['price'], errors='coerce')
    trades['qty'] = pd.to_numeric(trades['qty'], errors='coerce')
    trades['notional'] = trades['price'] * trades['qty']
    trades['signed_qty'] = np.where(trades['side'].eq('buy'), trades['qty'], -trades['qty'])
    trades['signed_notional'] = np.where(trades['side'].eq('buy'), trades['notional'], -trades['notional'])
    trades['minute'] = trades['recv_ts'].dt.floor('1min')

trades[['recv_seq', 'recv_ts', 'price', 'qty', 'side', 'signed_qty', 'signed_notional']].head() if not trades.empty else 'no trades'


## Join Book State To Trades

Use `merge_asof` on `recv_seq` so each trade gets the most recent replayed book state.


In [ ]:
if not books.empty and not trades.empty:
    book_cols = ['recv_seq', 'recv_ts', 'mid', 'spread', 'imbalance_l1', 'imbalance_l5', 'imbalance_l20']
    trades_with_book = pd.merge_asof(
        trades.sort_values('recv_seq'),
        books[book_cols].sort_values('recv_seq'),
        on='recv_seq',
        direction='backward',
        suffixes=('', '_book'),
    )
else:
    trades_with_book = pd.DataFrame()

trades_with_book.head() if not trades_with_book.empty else 'no joined trades'


## Example Statistical View: 1-Minute Aggregates

This produces a simple panel for exploratory analysis.


In [ ]:
if not books.empty:
    book_1m = (
        books.set_index('recv_ts')
        .resample('1min')
        .agg({
            'mid': ['first', 'last', 'mean'],
            'spread': ['mean', 'max'],
            'imbalance_l1': 'mean',
            'imbalance_l5': 'mean',
            'imbalance_l20': 'mean',
            'recv_seq': 'count',
        })
    )
    book_1m.columns = ['_'.join(c).strip('_') for c in book_1m.columns]
    book_1m['mid_ret_fwd_1m'] = np.log(book_1m['mid_last']).shift(-1) - np.log(book_1m['mid_last'])
else:
    book_1m = pd.DataFrame()

if not trades.empty:
    trade_1m = (
        trades.set_index('recv_ts')
        .resample('1min')
        .agg({
            'qty': 'sum',
            'notional': 'sum',
            'signed_qty': 'sum',
            'signed_notional': 'sum',
            'recv_seq': 'count',
        })
        .rename(columns={'recv_seq': 'trade_count'})
    )
else:
    trade_1m = pd.DataFrame()

panel_1m = book_1m.join(trade_1m, how='outer').sort_index()
panel_1m.head(10)


In [ ]:
# Simple correlation view (exploratory only)
cols = [
    'imbalance_l1_mean', 'imbalance_l5_mean', 'imbalance_l20_mean',
    'signed_qty', 'signed_notional', 'spread_mean', 'mid_ret_fwd_1m'
]
panel_1m[cols].dropna().corr() if not panel_1m.empty else pd.DataFrame()


## Plots


In [ ]:
if not panel_1m.empty:
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    panel_1m['mid_last'].plot(ax=axes[0], title='Mid Price (1m last)')
    panel_1m['spread_mean'].plot(ax=axes[1], title='Spread Mean (1m)')
    panel_1m['imbalance_l5_mean'].plot(ax=axes[2], title='Order Book Imbalance L5 (1m mean)')
    if 'signed_notional' in panel_1m:
        panel_1m['signed_notional'].plot(ax=axes[3], title='Signed Trade Notional (1m sum)')
    plt.tight_layout()
else:
    print('No panel data to plot')


## Next Ideas

- Use `segment_index` / `epoch_id` to exclude resync windows
- Build event-time bars instead of wall-clock bars
- Estimate impact curves by bucketed signed trade flow
- Compare features across venues using the same replay output schema
